# Resource-Aware Dispatch: Data Preparation
This notebook reproduces the preprocessing, alert-priority scoring, and allocation-input preparation completed so far. It does not yet run the allocation optimizer.

Alert requests, resource inventory, and dispatch assignments are synthetic. Resource coordinates are approximate DS-centre locations. Priority scores are a prototype heuristic, and candidate distances are straight-line estimates, not road travel times.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re
import numpy as np
import pandas as pd

print('pandas:', pd.__version__)

## 1. Set the Drive folder
Set `DATA_DIR` to the Google Drive folder containing all four workbooks. Update the example path if your folder has a different name. This notebook writes results into a `processed_data` subfolder and does not overwrite the source workbooks.

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/REPLACE_WITH_YOUR_FOLDER_NAME')
INPUT_FILES = {
    'alerts': 'Alert_Requests_5Districts.xlsx',
    'resources': 'Resource_Inventory_5Districts.xlsx',
    'assignments': 'Dispatch_Assignments_5Districts.xlsx',
    'population': 'GN_population_excel.xlsx',
}

missing_files = [name for name in INPUT_FILES.values() if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f'Could not find {missing_files} in {DATA_DIR}. Set DATA_DIR to the Drive folder containing the workbooks.'
    )

OUTPUT_DIR = DATA_DIR / 'processed_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Input folder:', DATA_DIR)
print('Output folder:', OUTPUT_DIR)

## 2. Inspect and load the workbooks
The three dispatch workbooks use their first row as headers. The population workbook has two header rows beginning at Excel row 4, so it is loaded separately.

In [ ]:
for key, filename in INPUT_FILES.items():
    workbook = pd.ExcelFile(DATA_DIR / filename)
    print(f'{filename}: sheets={workbook.sheet_names}')

alerts = pd.read_excel(DATA_DIR / INPUT_FILES['alerts'])
resources = pd.read_excel(DATA_DIR / INPUT_FILES['resources'])
assignments = pd.read_excel(DATA_DIR / INPUT_FILES['assignments'])
population_raw = pd.read_excel(
    DATA_DIR / INPUT_FILES['population'],
    sheet_name='Population',
    header=[3, 4],
)

for name, table in {
    'alerts': alerts,
    'resources': resources,
    'assignments': assignments,
    'population_raw': population_raw,
}.items():
    print(f'{name}: {table.shape}')
    display(table.head(3))

## 3. Standardize and validate source tables
This cell trims text, standardizes district and resource-type labels, converts dates and numeric fields, and filters population to Ratnapura, Kegalle, Kalutara, Badulla, and Nuwara Eliya.

In [ ]:
TARGET_DISTRICTS = {
    'ratnapura': 'Ratnapura',
    'kegalle': 'Kegalle',
    'kalutara': 'Kalutara',
    'badulla': 'Badulla',
    'nuwara eliya': 'Nuwara Eliya',
}

def clean_text(table):
    table = table.copy()
    for column in table.select_dtypes(include=['object', 'string']).columns:
        table[column] = table[column].astype('string').str.strip()
        table[column] = table[column].replace('', pd.NA)
    return table

def normalize_districts(table, columns):
    for column in columns:
        values = table[column].astype('string').str.strip()
        table[column] = values.str.casefold().map(TARGET_DISTRICTS).fillna(values)
    return table

def flatten_population_columns(table):
    direct_names = {
        'province code': 'province_code', 'province name': 'province_name',
        'district code': 'district_code', 'district name': 'district_name',
        'ds_division code': 'ds_division_code', 'ds_division name': 'ds_division_name',
        'gn_division code': 'gn_division_code', 'gn_division name': 'gn_division_name',
        'gn_division number': 'gn_division_number',
    }
    names = []
    for primary, secondary in table.columns:
        first = str(primary).replace('\n', ' ').strip().casefold()
        second = str(secondary).replace('\n', ' ').strip().casefold()
        if first in direct_names:
            names.append(direct_names[first])
        elif first in {'sex', 'age_group'}:
            prefix = 'sex' if first == 'sex' else 'age'
            suffix = re.sub(r'[^a-z0-9]+', '_', second).strip('_')
            names.append(f'{prefix}_{suffix}')
        else:
            names.append(f'{first}_{second}')
    table = table.copy()
    table.columns = names
    return table

alerts = normalize_districts(clean_text(alerts), ['district'])
resources = normalize_districts(clean_text(resources), ['district'])
assignments = normalize_districts(
    clean_text(assignments), ['origin_district', 'destination_district']
)
population = normalize_districts(
    clean_text(flatten_population_columns(population_raw)), ['district_name']
)

for table, column in [
    (alerts, 'requested_resource_type'),
    (resources, 'resource_type'),
    (assignments, 'resource_type'),
]:
    table[column] = table[column].str.upper()

for table, column in [
    (alerts, 'timestamp'), (resources, 'last_updated'),
    (assignments, 'assignment_time'), (assignments, 'estimated_arrival'),
    (assignments, 'status_time'),
]:
    table[column] = pd.to_datetime(table[column], errors='coerce')

numeric_columns = {
    'alerts': (alerts, ['confidence', 'latitude', 'longitude', 'affected_population', 'affected_households', 'requested_quantity']),
    'resources': (resources, ['latitude', 'longitude', 'capacity', 'offline_count', 'available_count']),
    'assignments': (assignments, ['assigned_quantity', 'destination_latitude', 'destination_longitude']),
}
for table, columns in numeric_columns.values():
    for column in columns:
        table[column] = pd.to_numeric(table[column], errors='coerce')

population_5 = population[population['district_name'].isin(TARGET_DISTRICTS.values())].copy()
print('Population rows retained:', len(population_5))
print('Districts:', sorted(population_5['district_name'].dropna().unique()))

In [ ]:
required = {
    'alerts': {'alert_id', 'district', 'GN_division', 'requested_resource_type', 'requested_quantity', 'confidence'},
    'resources': {'resource_id', 'resource_type', 'district', 'capacity', 'offline_count', 'available_count'},
    'assignments': {'assignment_id', 'alert_id', 'resource_id', 'resource_type', 'assigned_quantity'},
}
tables = {'alerts': alerts, 'resources': resources, 'assignments': assignments}
for name, expected in required.items():
    missing = expected - set(tables[name].columns)
    if missing:
        raise ValueError(f'{name} is missing columns: {sorted(missing)}')

if alerts.duplicated(['alert_id', 'requested_resource_type']).any():
    raise ValueError('Duplicate alert/resource-type request rows found')
if resources['resource_id'].isna().any() or resources['resource_id'].duplicated().any():
    raise ValueError('Resource IDs are missing or duplicated')
if assignments['assignment_id'].isna().any() or assignments['assignment_id'].duplicated().any():
    raise ValueError('Assignment IDs are missing or duplicated')
if (alerts['requested_quantity'] <= 0).any() or (assignments['assigned_quantity'] <= 0).any():
    raise ValueError('Request and assignment quantities must be positive')
if ((alerts['confidence'] < 0) | (alerts['confidence'] > 1)).any():
    raise ValueError('Alert confidence must be between 0 and 1')
if (resources['offline_count'] + resources['available_count'] > resources['capacity']).any():
    raise ValueError('Resource counts exceed stated capacity')

if not assignments['alert_id'].isin(alerts['alert_id']).all():
    raise ValueError('An assignment refers to an unknown alert')
if not assignments['resource_id'].isin(resources['resource_id']).all():
    raise ValueError('An assignment refers to an unknown resource')
resource_types = resources.set_index('resource_id')['resource_type']
if not assignments.apply(lambda row: row['resource_type'] == resource_types.loc[row['resource_id']], axis=1).all():
    raise ValueError('Assignment resource types do not match inventory')

request_totals = alerts.set_index(['alert_id', 'requested_resource_type'])['requested_quantity']
assigned_totals = assignments.groupby(['alert_id', 'resource_type'])['assigned_quantity'].sum()
for key, quantity in assigned_totals.items():
    if key not in request_totals.index or quantity > request_totals.loc[key]:
        raise ValueError(f'Assignment exceeds or lacks a matching request: {key}')

clean_tables = {
    'alerts_clean.csv': alerts,
    'resource_inventory_clean.csv': resources,
    'dispatch_assignments_clean.csv': assignments,
    'gn_population_5_districts_clean.csv': population_5,
}
for filename, table in clean_tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False, encoding='utf-8-sig')

validation_lines = [
    'Colab preprocessing validation: PASSED',
    'Synthetic inputs: alert requests, resource inventory, and assignments.',
    'Resource coordinates are approximate DS-centre locations.',
]
for name, table in tables.items():
    validation_lines.append(f'{name}: {len(table)} rows, {table.shape[1]} columns')
    missing_values = table.isna().sum()
    missing_values = missing_values[missing_values > 0]
    validation_lines.append(f'  Missing values: {missing_values.to_dict() if len(missing_values) else "none"}')
validation_lines.append(f'population_5_districts: {len(population_5)} rows')
(OUTPUT_DIR / 'preprocessing_report.txt').write_text('\n'.join(validation_lines), encoding='utf-8')

print('Source-table validation passed.')
print('Rows:', {name: len(table) for name, table in tables.items()})
print('Optional assigned_alert_id nulls:', int(resources['assigned_alert_id'].isna().sum()))
print('Cleaned tables saved to:', OUTPUT_DIR)

## 4. Compare demand with local availability
This is an aggregated, same-district snapshot comparison. It does not yet optimize assignments or consider moving resources between districts.

In [ ]:
demand_by_district = alerts.groupby(
    ['district', 'requested_resource_type'], as_index=False
).agg(
    requested_quantity=('requested_quantity', 'sum'),
    alert_count=('alert_id', 'nunique'),
)
supply_by_district = resources.groupby(
    ['district', 'resource_type'], as_index=False
).agg(
    available_quantity=('available_count', 'sum'),
    listed_capacity=('capacity', 'sum'),
    inventory_records=('resource_id', 'nunique'),
)
demand_vs_supply = demand_by_district.merge(
    supply_by_district,
    left_on=['district', 'requested_resource_type'],
    right_on=['district', 'resource_type'],
    how='left',
)
demand_vs_supply['available_quantity'] = demand_vs_supply['available_quantity'].fillna(0)
demand_vs_supply['listed_capacity'] = demand_vs_supply['listed_capacity'].fillna(0)
demand_vs_supply['shortage_quantity'] = (demand_vs_supply['requested_quantity'] - demand_vs_supply['available_quantity']).clip(lower=0)
demand_vs_supply['surplus_quantity'] = (demand_vs_supply['available_quantity'] - demand_vs_supply['requested_quantity']).clip(lower=0)
demand_vs_supply = demand_vs_supply.drop(columns=['resource_type'])
demand_vs_supply.to_csv(OUTPUT_DIR / 'district_resource_demand_vs_availability.csv', index=False)
display(demand_vs_supply.sort_values(['district', 'requested_resource_type']))

## 5. Score each unique alert
Priority is a prototype heuristic: 50% severity, 30% affected-population exposure relative to this alert batch, and 20% GN population share aged 0–14 or 65+. Population is joined using district, DS division, and GN division to avoid ambiguous GN names.

In [ ]:
severity_scores = {'critical': 1.0, 'high': 0.75, 'medium': 0.5, 'low': 0.25}
weights = {'severity_score': 0.50, 'exposure_score': 0.30, 'vulnerability_score': 0.20}
alert_fields = [
    'alert_id', 'timestamp', 'hazard_type', 'severity', 'confidence', 'district',
    'DS_division', 'GN_division', 'latitude', 'longitude', 'affected_population',
    'affected_households',
]
inconsistent = alerts.groupby('alert_id')[alert_fields[1:]].nunique(dropna=False).gt(1).any(axis=1)
if inconsistent.any():
    raise ValueError(f'Alert-level fields differ across request rows: {inconsistent[inconsistent].index.tolist()}')
unique_alerts = alerts[alert_fields].drop_duplicates('alert_id').copy()
unique_alerts['_district_key'] = unique_alerts['district'].astype('string').str.strip().str.casefold()
unique_alerts['_ds_key'] = unique_alerts['DS_division'].astype('string').str.strip().str.casefold()
unique_alerts['_gn_key'] = unique_alerts['GN_division'].astype('string').str.strip().str.casefold()
population_for_join = population_5[['district_name', 'ds_division_name', 'gn_division_name', 'age_total', 'age_0_14', 'age_65_and_above']].copy()
population_for_join['_district_key'] = population_for_join['district_name'].astype('string').str.strip().str.casefold()
population_for_join['_ds_key'] = population_for_join['ds_division_name'].astype('string').str.strip().str.casefold()
population_for_join['_gn_key'] = population_for_join['gn_division_name'].astype('string').str.strip().str.casefold()
join_keys = ['_district_key', '_ds_key', '_gn_key']
alert_locations = unique_alerts[join_keys].drop_duplicates()
population_for_alerts = population_for_join.merge(alert_locations, on=join_keys, how='inner')
if population_for_alerts.duplicated(join_keys).any():
    raise ValueError('An alert location matches multiple GN population rows')
if len(population_for_alerts) != len(alert_locations):
    raise ValueError('At least one alert location did not match GN population')

for column in ['age_total', 'age_0_14', 'age_65_and_above']:
    population_for_alerts[column] = pd.to_numeric(population_for_alerts[column], errors='coerce')
if population_for_alerts[['age_total', 'age_0_14', 'age_65_and_above']].isna().any().any():
    raise ValueError('GN age fields contain missing or nonnumeric values')
if (population_for_alerts['age_total'] <= 0).any():
    raise ValueError('GN total population must be positive')
if (population_for_alerts['age_0_14'] + population_for_alerts['age_65_and_above'] > population_for_alerts['age_total']).any():
    raise ValueError('Vulnerable age counts exceed GN total population')

unique_alerts['severity_score'] = unique_alerts['severity'].astype('string').str.strip().str.casefold().map(severity_scores)
if unique_alerts['severity_score'].isna().any():
    raise ValueError('Found an unmapped severity level')
unique_alerts['affected_population'] = pd.to_numeric(unique_alerts['affected_population'], errors='coerce')
if unique_alerts['affected_population'].isna().any() or (unique_alerts['affected_population'] < 0).any():
    raise ValueError('Affected population must be present and nonnegative')
max_exposure = unique_alerts['affected_population'].max()
unique_alerts['exposure_score'] = unique_alerts['affected_population'] / max_exposure if max_exposure > 0 else 0.0

priority_scores = unique_alerts.merge(
    population_for_alerts[join_keys + ['age_total', 'age_0_14', 'age_65_and_above']],
    on=join_keys, how='left', validate='many_to_one'
)
priority_scores['vulnerable_population_share'] = (priority_scores['age_0_14'] + priority_scores['age_65_and_above']) / priority_scores['age_total']
priority_scores['vulnerability_score'] = priority_scores['vulnerable_population_share']
priority_scores['priority_score'] = sum(priority_scores[name] * weight for name, weight in weights.items())
priority_scores['priority_rank'] = priority_scores['priority_score'].rank(method='first', ascending=False).astype(int)
priority_scores = priority_scores.drop(columns=join_keys).sort_values(['priority_rank', 'alert_id'])

score_fields = ['alert_id', 'severity_score', 'exposure_score', 'vulnerability_score', 'priority_score', 'priority_rank']
prioritized_requests = alerts.merge(priority_scores[score_fields], on='alert_id', how='left', validate='many_to_one')
if prioritized_requests['priority_score'].isna().any():
    raise ValueError('A request row did not receive an alert priority score')
priority_scores.to_csv(OUTPUT_DIR / 'alert_priority_scores.csv', index=False)
prioritized_requests.to_csv(OUTPUT_DIR / 'alert_requests_prioritized.csv', index=False)
display(priority_scores[['priority_rank', 'alert_id', 'district', 'GN_division', 'priority_score']].head(10))
print('Unique alerts scored:', len(priority_scores), '| prioritized request rows:', len(prioritized_requests))

In [ ]:
priority_methodology = [
    'Prototype alert-priority scoring methodology',
    'One score is computed per unique alert; each resource-request row inherits that score.',
    'Severity mapping: Critical=1.00, High=0.75, Medium=0.50, Low=0.25.',
    'Exposure score = affected_population / maximum affected_population in this alert batch.',
    'Vulnerability score = (GN population aged 0-14 + aged 65+) / GN total population.',
    'Priority score = 0.50*severity + 0.30*exposure + 0.20*vulnerability.',
    'This is a transparent prototype heuristic, not a trained or empirically validated model.',
    'Alert and resource data are synthetic; GN population data is provisional.',
    'The exposure score is relative to this alert batch and may change when alerts change.',
]
(OUTPUT_DIR / 'priority_scoring_methodology.txt').write_text(
    '\n'.join(priority_methodology), encoding='utf-8'
)
print('Priority methodology saved.')

## 6. Prepare allocation candidates
Each request is paired with all available inventory records of the same resource type. Cross-district resources are included. The optimizer must later enforce shared resource capacities across requests; candidate rows are not assignments.

In [ ]:
available_supply = resources[
    pd.to_numeric(resources['available_count'], errors='coerce').fillna(0) > 0
].copy()
requested_types = set(
    prioritized_requests['requested_resource_type'].dropna().str.upper()
)
available_types = set(available_supply['resource_type'].dropna().str.upper())
if requested_types - available_types:
    raise ValueError(
        f'Requested resource types with no available inventory: {sorted(requested_types - available_types)}'
    )

supply_for_join = available_supply.rename(columns={
    'resource_type': 'candidate_resource_type',
    'district': 'resource_district',
    'DS_division': 'resource_DS_division',
    'GN_division': 'resource_GN_division',
    'latitude': 'resource_latitude',
    'longitude': 'resource_longitude',
    'available_count': 'available_quantity',
    'status': 'resource_status',
    'last_updated': 'inventory_last_updated',
})
candidates = prioritized_requests.merge(
    supply_for_join,
    left_on='requested_resource_type',
    right_on='candidate_resource_type',
    how='left',
)
candidates = candidates.dropna(subset=['resource_id']).copy()
candidates['is_same_district'] = (
    candidates['district'].str.casefold()
    == candidates['resource_district'].str.casefold()
)

earth_radius_km = 6371.0088
lat1 = np.radians(pd.to_numeric(candidates['latitude']).to_numpy())
lon1 = np.radians(pd.to_numeric(candidates['longitude']).to_numpy())
lat2 = np.radians(pd.to_numeric(candidates['resource_latitude']).to_numpy())
lon2 = np.radians(pd.to_numeric(candidates['resource_longitude']).to_numpy())
delta_lat = lat2 - lat1
delta_lon = lon2 - lon1
hav = (
    np.sin(delta_lat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
)
candidates['straight_line_distance_km'] = (
    2 * earth_radius_km * np.arcsin(np.sqrt(np.clip(hav, 0, 1)))
)
candidates['candidate_rank_by_distance'] = candidates.groupby(
    ['alert_id', 'requested_resource_type']
)['straight_line_distance_km'].rank(method='first').astype(int)
candidates['distance_basis'] = 'Straight-line estimate; not road travel distance'

request_pairs = set(zip(
    prioritized_requests['alert_id'],
    prioritized_requests['requested_resource_type'],
))
candidate_pairs = set(zip(
    candidates['alert_id'], candidates['requested_resource_type']
))
if request_pairs - candidate_pairs:
    raise ValueError(
        f'Requests without compatible available candidates: {sorted(request_pairs - candidate_pairs)}'
    )
if candidates.duplicated(
    ['alert_id', 'requested_resource_type', 'resource_id']
).any():
    raise ValueError('Duplicate alert/resource candidate pairs found')

candidates = candidates.sort_values(
    ['priority_rank', 'candidate_rank_by_distance', 'alert_id']
).reset_index(drop=True)
available_supply.to_csv(OUTPUT_DIR / 'available_resource_supply.csv', index=False)
candidates.to_csv(OUTPUT_DIR / 'allocation_candidates.csv', index=False)
print('Request rows:', len(prioritized_requests))
print('Available inventory records:', len(available_supply))
print('Compatible candidate pairs:', len(candidates))
print('Same-district pairs:', int(candidates['is_same_district'].sum()))
print('Cross-district pairs:', int((~candidates['is_same_district']).sum()))
print('Outputs saved to:', OUTPUT_DIR)

In [ ]:
prioritized_requests.to_csv(
    OUTPUT_DIR / 'allocation_requests.csv', index=False
)
allocation_notes = [
    'Allocation input preparation',
    'Requests are synthetic and inherit prototype priority scores.',
    'Supply is the available_count snapshot from the synthetic inventory workbook.',
    'Candidates pair each request with available inventory of the same resource type.',
    'Cross-district resources are included; the optimizer must enforce shared capacities.',
    'Straight-line distance is a proximity proxy, not road travel time or a safe route.',
    'Resource coordinates are approximate DS-centre locations.',
    f'Request rows: {len(prioritized_requests)}',
    f'Available inventory records: {len(available_supply)}',
    f'Compatible candidate pairs: {len(candidates)}',
    f'Same-district candidate pairs: {int(candidates["is_same_district"].sum())}',
    f'Cross-district candidate pairs: {int((~candidates["is_same_district"]).sum())}',
]
(OUTPUT_DIR / 'allocation_input_notes.txt').write_text(
    '\n'.join(allocation_notes), encoding='utf-8'
)
print('Allocation request table and notes saved.')